# Module 18 — LCEL and Runnables, then the same job in three plain lines

**THE ONE IDEA:** LCEL has **one** good idea — everything implements the same interface
(`.invoke` / `.batch` / `.stream`), so `|` can compose anything with anything.

Everything else LangChain offers is integrations and legacy.

Note what this notebook does **not** import: `langchain_openai`. The steps are
`RunnableLambda`s wrapping the raw SDK call you already wrote in module 02. That is the
honest picture — **a framework composes your code; it does not supply the capability.**


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
from _providers import get_client
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

client, MODEL, _ = get_client("openai")

def llm(prompt: str) -> str:
    r = client.chat.completions.create(model=MODEL, max_tokens=250,
                                       messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content.strip()

# Three plain functions. Nothing framework-shaped about them.
template = lambda d: f"Explain {d['topic']} for a UK mortgage in 2 sentences."
shorten  = lambda t: " ".join(t.split()[:28]) + "..."

prompt_r, llm_r, short_r = RunnableLambda(template), RunnableLambda(llm), RunnableLambda(shorten)
print("three Runnables, each wrapping a plain callable")

## The pipe

`|` builds a new Runnable that runs them in sequence. The composed object gets
`.invoke`, `.batch` and `.stream` **for free** — that is what the protocol buys.

In [ ]:
chain = prompt_r | llm_r | short_r

print("INVOKE:", chain.invoke({"topic": "the early repayment charge"}))

print("\nBATCH (concurrent, one call site):")
for out in chain.batch([{"topic": "LTV limits"}, {"topic": "proof of income"}]):
    print("  -", out[:88])

## RunnableParallel — fan out, keep the input

Module 06 did this with `ThreadPoolExecutor`. Same shape, declared instead of coded.

In [ ]:
fan = RunnableParallel(
    erc=RunnableLambda(lambda d: llm(f"ERC rules for {d['product']}? One sentence.")),
    ltv=RunnableLambda(lambda d: llm(f"LTV limit for {d['product']}? One sentence.")),
    original=RunnablePassthrough(),          # carries the input through untouched
)
out = fan.invoke({"product": "a 5-year fixed first-time-buyer mortgage"})
for k, v in out.items():
    print(f"  {k:9} {str(v)[:74]}")

## The same job, three plain lines

In [ ]:
def plain(topic):
    return shorten(llm(template({"topic": topic})))

print("LCEL  :", chain.invoke({"topic": "the standard variable rate"})[:80])
print("PLAIN :", plain("the standard variable rate")[:80])

print("""
LESSON - identical output. So what did LCEL actually buy?

  BOUGHT   one interface everywhere: .invoke / .batch / .stream / .ainvoke, and
           async + concurrency for free on anything you compose
  BOUGHT   ~100 pre-built integrations you do not have to write
  BOUGHT   a uniform callback surface, so LangSmith traces every step

  COST     the prompt is now behind an abstraction - run set_debug(True) or you
           do not know what the model actually received
  COST     100-300 ms per chain, which is real on a sub-second SLA
  COST     a large dependency tree and breaking changes across minor versions

The senior position, and it is a real hiring signal: use plain functions for
composition, because functions already compose. Reach for a framework when you
need something genuinely hard to hand-roll - CHECKPOINTING, HUMAN-IN-THE-LOOP,
REPLAY. Those are modules 19 and 20, and that is where LangGraph earns its place.

Do NOT reach for AgentExecutor. It is legacy - a black-box while loop. Module 19
shows what replaced it.""")

---

**Next:** `19_langgraph_minimal_agent.ipynb` — module 08 as a state machine.